# GPUPerf-3B: Train & Test the Real Model

This notebook runs **Stage 1 (QLoRA SFT)** and **Stage 2 (DPO Preference Alignment)** on `Qwen2.5-Coder-3B-Instruct` using **100% real physical GPU measurements** harvested across NVIDIA hardware.

### Pipeline Overview:
1. **Base Model**: `Qwen/Qwen2.5-Coder-3B-Instruct` (4-bit NF4 via Unsloth / PEFT)
2. **Training Data**: Real physical executions measured on Tesla T4 (`sm_75`) and NVIDIA L4 (`sm_89`)
3. **Inference Verification**: Live test demonstrating hardware-conditional kernel optimization.

In [ ]:
# 1. Verify Attached GPU
!nvidia-smi

In [ ]:
# 2. Fast Install Unsloth & Modern Fine-Tuning Stack
%%capture
!pip install --no-deps "xformers<0.0.29" triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 3. Clone Repository & Setup Dataset
import os
if not os.path.exists("GPUPerf"):;
    !git clone https://github.com/sarimahsan/GPUPerf.git
%cd GPUPerf
!git pull
!ls -lh ./data

In [ ]:
# 4. Load Base Model: Qwen2.5-Coder-3B-Instruct (4-bit QLoRA)
import torch
from unsloth import FastLanguageModel

max_seq_length = 3072
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print("Model loaded and LoRA adapters initialized!")

In [ ]:
# 5. Load Real Multi-GPU SFT Dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="./data/sft_multi_gpu.jsonl", split="train")
print(f"Loaded {len(dataset)} verified physical training samples!")
print("Sample hardware tags:", set(dataset["metadata"]["device"]))

In [ ]:
# 6. Train Stage 1: Supervised Fine-Tuning (SFT)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="messages",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./checkpoints/gpuperf_sft_model",
        report_to="none"
    ),
)

print("Starting SFT training...")
trainer_stats = trainer.train()
print("SFT Training Finished Successfully!")

In [ ]:
# 7. Save Fine-Tuned Adapters
output_model_dir = "./checkpoints/gpuperf_3b_pilot"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Saved GPUPerf-3B adapter to {output_model_dir}")

In [ ]:
# 8. Interactive Real Model Inference & Testing
# Enable fast native inference
FastLanguageModel.for_inference(model)

test_prompt = """Here is a TRITON GPU kernel and its hardware profiling report from NVIDIA Nsight Compute:

```triton
# Unoptimized GEMM baseline with 64x64 tiles
BLOCK_M = 64
BLOCK_N = 64
BLOCK_K = 32
num_warps = 4
num_stages = 2
```

[PROFILER REPORT]
Device: NVIDIA L4 (sm_89, 58 SMs, 300.0 GB/s DRAM)
Kernel: gemm_benchmark
Measured Latency: 555.2 us
Occupancy: Achieved 35.0%
Bottleneck: SuboptimalTileOrOccupancy
[/PROFILER REPORT]

Analyze the profiler metrics and write an optimized kernel for NVIDIA L4."""

inputs = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": "You are GPUPerf, a specialized GPU systems optimization engineer."},
        {"role": "user", "content": test_prompt}
    ],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=1024, temperature=0.2)
print("=== Model Optimization Output ===")
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))